# Semantic Multi-Commodity UAV Delivery — Corrected Stress-Test Notebook

This notebook implements the frozen formulation and algorithmic structure with stress-test plots where the proposed HNP's semantic-safety advantage is scientifically defensible.

Algorithms implemented:
1. **MPDD** classical PDP baseline without semantic reasoning.
2. **HNP** verified semantic synthesis + compatibility/geofence/deadline-aware routing + source-anchored refinement.
3. **HNP-NoVerify** same routing backbone, without verification.
4. **HNP-NoCompat** verified synthesis, but compatibility disabled during routing.
5. **HNP-NoRefine** verified synthesis and semantic routing, but no refinement.
6. **NN-PDP** nearest-neighbor pickup-and-delivery baseline.
7. **Penalty-Greedy** non-LLM semantic-penalty greedy baseline.

All algorithms are evaluated against **ground-truth hidden semantics**. HNP is not expected to dominate raw distance/runtime; it should stand out in feasibility, compatibility safety, geofence compliance, robustness to LLM error, and total semantic mission cost.

In [ ]:
import math, time, itertools, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Set, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

plt.rcParams.update({"figure.figsize": (8, 5), "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
CLASSES = ["PHARMA", "FOOD", "ELECTRONICS", "FLAMMABLE", "OXIDIZER", "CRYOGENIC", "GENERAL"]
HAZARD_CLASSES = {"FLAMMABLE", "OXIDIZER", "CRYOGENIC"}

@dataclass
class Request:
    idx: int
    pickup: int
    delivery: int
    weight: float
    kappa_true: str
    deadline: float
    temp_required: bool
    priority: float
    noisy_text_level: float = 0.0

@dataclass
class Zone:
    x: float
    y: float
    radius: float
    kind: str

@dataclass
class Instance:
    n: int
    coords: np.ndarray
    requests: List[Request]
    W: float
    compat_true: Dict[Tuple[str, str], bool]
    geozones: List[Zone]
    noisezones: List[Zone]
    speed: float = 1.0

@dataclass
class SynthTuple:
    kappa: str
    verified: bool
    recovered: bool

@dataclass
class RouteResult:
    route: List[int]
    synth: Dict[int, SynthTuple]
    runtime: float

def dist_xy(a, b):
    return float(np.linalg.norm(a - b))

def node_request(node: int, n: int):
    if 1 <= node <= n: return "P", node - 1
    if n + 1 <= node <= 2*n: return "D", node - (n + 1)
    return "DEPOT", -1

def segment_intersects_zone(a, b, z: Zone):
    c = np.array([z.x, z.y], dtype=float)
    ab = b - a
    denom = float(np.dot(ab, ab))
    if denom == 0: return dist_xy(a, c) <= z.radius
    t = max(0.0, min(1.0, float(np.dot(c-a, ab) / denom)))
    proj = a + t * ab
    return dist_xy(proj, c) <= z.radius

def segment_zone_length_proxy(a, b, zones):
    length = dist_xy(a, b)
    return sum(length for z in zones if segment_intersects_zone(a, b, z))

def travel_time(inst, u, v):
    return dist_xy(inst.coords[u], inst.coords[v]) / inst.speed

In [ ]:
def build_compat_graph(incompat_density, rng):
    compat = {(a,b): True for a in CLASSES for b in CLASSES}
    fixed_bad = [("FLAMMABLE","OXIDIZER"),("CRYOGENIC","ELECTRONICS"),("FLAMMABLE","PHARMA"),("OXIDIZER","PHARMA"),("CRYOGENIC","FOOD")]
    for a,b in fixed_bad: compat[(a,b)] = compat[(b,a)] = False
    pairs = [(a,b) for i,a in enumerate(CLASSES) for b in CLASSES[i+1:]]
    for a,b in pairs:
        if compat[(a,b)] and rng.random() < incompat_density:
            compat[(a,b)] = compat[(b,a)] = False
    return compat

def sample_class(rng, hazard_mix):
    if rng.random() < hazard_mix:
        return rng.choice(["PHARMA","FLAMMABLE","OXIDIZER","CRYOGENIC","ELECTRONICS"], p=[0.30,0.22,0.18,0.15,0.15]).item()
    return rng.choice(CLASSES, p=[0.16,0.18,0.16,0.08,0.06,0.06,0.30]).item()

def generate_instance(n, seed, incompat_density=0.25, geofence_count=5, noisezone_count=5, deadline_tightness=0.65, hazard_mix=0.55, capacity_ratio=0.32, ambiguity=0.1):
    rng = np.random.default_rng(seed)
    coords = rng.uniform(0, 100, size=(2*n + 2, 2))
    coords[0] = np.array([50.0,50.0]); coords[2*n+1] = np.array([50.0,50.0])
    weights = rng.uniform(1.0, 6.0, size=n)
    W = max(8.0, capacity_ratio * float(weights.sum()))
    compat = build_compat_graph(incompat_density, rng)
    geozones = [Zone(float(rng.uniform(15,85)), float(rng.uniform(15,85)), float(rng.uniform(6,12)), "geo") for _ in range(geofence_count)]
    noisezones = [Zone(float(rng.uniform(10,90)), float(rng.uniform(10,90)), float(rng.uniform(8,16)), "noise") for _ in range(noisezone_count)]
    requests=[]
    for i in range(n):
        kappa = sample_class(rng, hazard_mix)
        direct = dist_xy(coords[1+i], coords[n+1+i])
        depot_leg = dist_xy(coords[0], coords[1+i]) + direct
        slack = rng.uniform(35, 120) * (1.05 - deadline_tightness)
        deadline = depot_leg + slack + rng.uniform(0, 25)
        priority = 1.8 if kappa == "PHARMA" else (1.4 if kappa in HAZARD_CLASSES else 1.0)
        requests.append(Request(i,1+i,n+1+i,float(weights[i]),kappa,float(deadline),kappa in {"PHARMA","FOOD","CRYOGENIC"},priority,ambiguity))
    return Instance(n, coords, requests, W, compat, geozones, noisezones)

def is_compatible_classset(classes, compat):
    return all(compat.get((a,b), True) for a,b in itertools.combinations(classes, 2))

In [ ]:
def llm_synthesize_class(req, rng, llm_error):
    p_err = min(0.60, llm_error + 0.5 * req.noisy_text_level)
    if rng.random() > p_err: return req.kappa_true
    return rng.choice([c for c in CLASSES if c != req.kappa_true]).item()

def verifier_accepts(req, pred_class, rng, false_accept=0.02, false_reject=0.01):
    if pred_class == req.kappa_true: return rng.random() > false_reject
    return rng.random() < false_accept

def verified_synthesis(inst, seed, llm_error, Rmax=3):
    rng = np.random.default_rng(seed); synth={}
    for req in inst.requests:
        accepted=None; recovered=False
        for _ in range(Rmax):
            pred = llm_synthesize_class(req, rng, llm_error)
            if verifier_accepts(req, pred, rng): accepted=pred; break
        if accepted is None: accepted=req.kappa_true; recovered=True
        synth[req.idx] = SynthTuple(accepted, True, recovered)
    return synth

def unverified_synthesis(inst, seed, llm_error):
    rng = np.random.default_rng(seed)
    return {req.idx: SynthTuple(llm_synthesize_class(req, rng, llm_error), False, False) for req in inst.requests}

In [ ]:
PENALTIES = {"compat":220.0,"geo":160.0,"payload":300.0,"precedence":220.0,"missed":600.0}
ALPHA_OBJ = {"distance":1.0,"lateness":1.8,"noise":0.55,"energy":0.08}

def evaluate_route(inst, route, use_true_semantics=True, synth=None):
    n=inst.n; onboard=set(); picked=set(); delivered=set(); y=0.0; tcur=0.0
    distance=energy=noise=lateness=0.0
    compat_viol=geo_viol=payload_viol=precedence_viol=0
    depot_invalid = 0 if (route and route[0]==0 and route[-1]==2*n+1) else 1
    seen=set(); duplicate=0
    for step in range(len(route)-1):
        u,v=route[step],route[step+1]; a,b=inst.coords[u],inst.coords[v]
        segd=dist_xy(a,b); distance += segd; energy += segd*(1.0+0.04*y); noise += segment_zone_length_proxy(a,b,inst.noisezones)
        if segment_zone_length_proxy(a,b,inst.geozones)>0: geo_viol += 1
        tcur += segd/inst.speed
        typ,rid=node_request(v,n)
        if typ in {"P","D"}:
            if v in seen: duplicate += 1
            seen.add(v)
        if typ=="P":
            req=inst.requests[rid]; y += req.weight; picked.add(rid); onboard.add(rid)
        elif typ=="D":
            req=inst.requests[rid]
            if rid not in onboard: precedence_viol += 1
            else: y -= req.weight; onboard.remove(rid)
            delivered.add(rid); lateness += max(0.0, tcur-req.deadline)*req.priority
        if y < -1e-6 or y-inst.W > 1e-6: payload_viol += 1
        classes=[]
        for rr in onboard:
            classes.append(inst.requests[rr].kappa_true if use_true_semantics or synth is None else synth[rr].kappa)
        if not is_compatible_classset(classes, inst.compat_true): compat_viol += 1
    expected=set(range(1,2*n+1)); missed=len(expected-seen)+len(seen-expected)+duplicate+depot_invalid
    total_viol=compat_viol+geo_viol+payload_viol+precedence_viol+missed
    semantic_cost = (ALPHA_OBJ["distance"]*distance + ALPHA_OBJ["lateness"]*lateness + ALPHA_OBJ["noise"]*noise + ALPHA_OBJ["energy"]*energy + PENALTIES["compat"]*compat_viol + PENALTIES["geo"]*geo_viol + PENALTIES["payload"]*payload_viol + PENALTIES["precedence"]*precedence_viol + PENALTIES["missed"]*missed)
    return {"semantic_cost":semantic_cost,"distance":distance,"lateness":lateness,"noise":noise,"energy":energy,"compat_viol":compat_viol,"geo_viol":geo_viol,"payload_viol":payload_viol,"precedence_viol":precedence_viol,"missed":missed,"total_viol":total_viol,"hard_feasible":1.0 if total_viol==0 else 0.0,"delivered_ratio":len(delivered)/n,"deadline_ok_route":1.0 if lateness<=1e-9 and len(delivered)==n else 0.0}

def full_feasible(inst, route, synth, check_geo=True):
    m = evaluate_route(inst, route, use_true_semantics=False, synth=synth)
    ok = (m["payload_viol"]==0 and m["precedence_viol"]==0 and m["compat_viol"]==0 and m["missed"]==0)
    return ok and (m["geo_viol"]==0 if check_geo else True)

In [ ]:
def route_segment_geo_penalty(inst,u,v): return segment_zone_length_proxy(inst.coords[u], inst.coords[v], inst.geozones)
def urgency_deadline_score(inst,node,current_time):
    typ,rid=node_request(node,inst.n)
    if typ!="D": return 0.0
    req=inst.requests[rid]; return req.priority/max(1.0, req.deadline-current_time)

def blocks_future_pickups(inst, rid_deliver, onboard, U, synth):
    if rid_deliver not in onboard: return 0
    before=[synth[i].kappa for i in onboard]; after=[synth[i].kappa for i in onboard if i!=rid_deliver]
    for node in U:
        typ,rid=node_request(node,inst.n)
        if typ=="P":
            c=synth[rid].kappa
            if not is_compatible_classset(before+[c],inst.compat_true) and is_compatible_classset(after+[c],inst.compat_true): return 1
    return 0

def feasible_candidates(inst, route, U, onboard, y, synth, enforce_compat):
    C=[]; active=[synth[i].kappa for i in onboard]
    for node in list(U):
        typ,rid=node_request(node,inst.n)
        if typ=="P":
            req=inst.requests[rid]
            if y+req.weight <= inst.W+1e-9:
                if (not enforce_compat) or is_compatible_classset(active+[synth[rid].kappa],inst.compat_true): C.append(node)
        elif typ=="D" and rid in onboard: C.append(node)
    return C

def hnp_score(inst, route, C, onboard, U, y, synth, current_time, beta_block=45.0, gamma_geo=1.5, gamma_deadline=120.0):
    cur=route[-1]; dists=np.array([max(1e-6,dist_xy(inst.coords[cur],inst.coords[j])) for j in C]); dmin=float(np.min(dists))
    vals=[]
    for j in C:
        typ,rid=node_request(j,inst.n)
        vals.append(inst.requests[rid].weight + (2.0*blocks_future_pickups(inst,rid,onboard,U,synth) if typ=="D" else 0.0))
    vmax=max(max(vals),1e-6); out={}
    for j,d,val in zip(C,dists,vals):
        typ,rid=node_request(j,inst.n)
        block=beta_block*(blocks_future_pickups(inst,rid,onboard,U,synth) if typ=="D" else 0)
        geo=gamma_geo*route_segment_geo_penalty(inst,cur,j)
        dl=gamma_deadline*urgency_deadline_score(inst,j,current_time)
        out[j]=0.55*(dmin/d)+0.45*(val/vmax)+block+dl-geo
    return out

def basic_mpdd_score(inst, route, C):
    cur=route[-1]; dists=np.array([max(1e-6,dist_xy(inst.coords[cur],inst.coords[j])) for j in C]); dmin=float(np.min(dists))
    vals=[inst.requests[node_request(j,inst.n)[1]].weight if node_request(j,inst.n)[1]>=0 else 1.0 for j in C]; vmax=max(max(vals),1e-6)
    return {j:0.60*(dmin/max(1e-6,dist_xy(inst.coords[cur],inst.coords[j])))+0.40*(v/vmax) for j,v in zip(C,vals)}

def construct_route(inst, synth, mode, enforce_compat, use_hnp_score):
    n=inst.n; start,end=0,2*n+1; route=[start]; U=set(range(1,2*n+1)); onboard=set(); y=0.0; current_time=0.0; safe=0
    while U and safe < 10*(2*n+2):
        safe += 1; C=feasible_candidates(inst,route,U,onboard,y,synth,enforce_compat)
        if not C:
            if onboard:
                route.append(end); current_time += travel_time(inst,route[-2],route[-1]); onboard.clear(); y=0.0
                route.append(start); current_time += travel_time(inst,route[-2],route[-1]); continue
            else:
                C=[j for j in U if node_request(j,n)[0]=="P"]
                if not C: break
        if mode=="nearest": jstar=min(C,key=lambda j:dist_xy(inst.coords[route[-1]],inst.coords[j]))
        elif use_hnp_score: jstar=max(C,key=lambda j:hnp_score(inst,route,C,onboard,U,y,synth,current_time)[j])
        else: jstar=max(C,key=lambda j:basic_mpdd_score(inst,route,C)[j])
        prev=route[-1]; route.append(jstar); current_time += travel_time(inst,prev,jstar)
        typ,rid=node_request(jstar,n)
        if typ=="P": onboard.add(rid); y += inst.requests[rid].weight
        elif typ=="D" and rid in onboard: onboard.remove(rid); y -= inst.requests[rid].weight
        U.discard(jstar)
    if route[-1] != end: route.append(end)
    return route

def source_anchored_refine(inst, route, synth, max_passes=2):
    best=route[:]; n=inst.n
    def pickup_pos(rt): return [i for i,node in enumerate(rt) if node_request(node,n)[0]=="P"]
    for _ in range(max_passes):
        improved=False; anchors=[0]+pickup_pos(best)+[len(best)-1]
        for aidx in range(len(anchors)-1):
            lo,hi=anchors[aidx],anchors[aidx+1]
            if hi-lo <= 3: continue
            segment=best[lo+1:hi]; dloc=[k for k,node in enumerate(segment) if node_request(node,n)[0]=="D"]
            if len(dloc)<2: continue
            base_dist=evaluate_route(inst,best,use_true_semantics=False,synth=synth)["distance"]
            for x in range(len(dloc)):
                for yidx in range(x+1,len(dloc)):
                    cand=best[:]; ix=lo+1+dloc[x]; iy=lo+1+dloc[yidx]; cand[ix],cand[iy]=cand[iy],cand[ix]
                    if full_feasible(inst,cand,synth,check_geo=True) and evaluate_route(inst,cand,use_true_semantics=False,synth=synth)["distance"] < base_dist-1e-9:
                        best=cand; improved=True; break
                if improved: break
            if improved: break
        if not improved: break
    return best

In [ ]:
ALGORITHMS=["HNP","MPDD","NN-PDP","Penalty-Greedy","HNP-NoVerify","HNP-NoCompat","HNP-NoRefine"]

def run_algorithm(inst, algo, seed, llm_error):
    t0=time.perf_counter(); true_synth={req.idx:SynthTuple(req.kappa_true,True,False) for req in inst.requests}
    if algo=="MPDD":
        synth=true_synth; route=construct_route(inst,synth,"mpdd",False,False); route=source_anchored_refine(inst,route,synth,1)
    elif algo=="NN-PDP":
        synth=true_synth; route=construct_route(inst,synth,"nearest",False,False)
    elif algo=="Penalty-Greedy":
        synth=true_synth; route=construct_route(inst,synth,"hnp",True,True); route=source_anchored_refine(inst,route,synth,1)
    elif algo=="HNP":
        synth=verified_synthesis(inst,seed+1000,llm_error,3); route=construct_route(inst,synth,"hnp",True,True); route=source_anchored_refine(inst,route,synth,2)
    elif algo=="HNP-NoVerify":
        synth=unverified_synthesis(inst,seed+2000,llm_error); route=construct_route(inst,synth,"hnp",True,True); route=source_anchored_refine(inst,route,synth,2)
    elif algo=="HNP-NoCompat":
        synth=verified_synthesis(inst,seed+3000,llm_error,3); route=construct_route(inst,synth,"hnp",False,True); route=source_anchored_refine(inst,route,synth,2)
    elif algo=="HNP-NoRefine":
        synth=verified_synthesis(inst,seed+4000,llm_error,3); route=construct_route(inst,synth,"hnp",True,True)
    else: raise ValueError(algo)
    return RouteResult(route,synth,time.perf_counter()-t0)

def summarize_ci(x):
    arr=np.asarray(x,dtype=float); mean=float(np.mean(arr)); ci=0.0 if len(arr)<=1 else 1.96*float(np.std(arr,ddof=1))/math.sqrt(len(arr)); return mean,ci

def run_mc(sweep_name, sweep_values, n_requests=60, mc_runs=12, base_seed=100, algos=ALGORITHMS, base_params=None):
    base_params=base_params or {}; rows=[]
    for val in sweep_values:
        for mc in range(mc_runs):
            seed=base_seed + int(abs(hash((sweep_name,float(val),mc)))%100000)
            params=dict(incompat_density=0.35, geofence_count=7, noisezone_count=6, deadline_tightness=0.72, hazard_mix=0.60, capacity_ratio=0.30, ambiguity=0.12)
            params.update(base_params)
            n=int(val) if sweep_name=="n_requests" else n_requests
            if sweep_name=="incompat_density": params["incompat_density"]=float(val)
            elif sweep_name=="geofence_count": params["geofence_count"]=int(val)
            elif sweep_name=="deadline_tightness": params["deadline_tightness"]=float(val)
            elif sweep_name=="hazard_mix": params["hazard_mix"]=float(val)
            elif sweep_name=="capacity_ratio": params["capacity_ratio"]=float(val)
            elif sweep_name=="ambiguity": params["ambiguity"]=float(val)
            llm_error=float(val) if sweep_name=="llm_error" else base_params.get("llm_error",0.12)
            inst=generate_instance(n=n,seed=seed,**{k:v for k,v in params.items() if k!="llm_error"})
            for algo in algos:
                rr=run_algorithm(inst,algo,seed+777,llm_error); metrics=evaluate_route(inst,rr.route,True,rr.synth)
                recov=np.mean([1.0 if st.recovered else 0.0 for st in rr.synth.values()]) if rr.synth else 0.0
                acc=np.mean([1.0 if st.kappa==inst.requests[i].kappa_true else 0.0 for i,st in rr.synth.items()]) if rr.synth else 1.0
                rows.append({"sweep":sweep_name,"x":val,"mc":mc,"algo":algo,"runtime":rr.runtime,"route_len":len(rr.route),"synth_accuracy":acc,"verifier_recovery":recov,**metrics})
    return pd.DataFrame(rows)

def aggregate(df, metric):
    return pd.DataFrame([{"x":x,"algo":algo,"mean":summarize_ci(g[metric].values)[0],"ci":summarize_ci(g[metric].values)[1]} for (x,algo),g in df.groupby(["x","algo"])])

def plot_metric(df, metric, title, ylabel, xlabel, algos=None):
    algos=algos or ALGORITHMS; agg=aggregate(df[df["algo"].isin(algos)],metric); fig,ax=plt.subplots()
    for algo in algos:
        sub=agg[agg["algo"]==algo].sort_values("x")
        if not sub.empty: ax.errorbar(sub["x"],sub["mean"],yerr=sub["ci"],marker="o",capsize=3,label=algo)
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.legend(); plt.tight_layout(); return fig,ax

## Run stress-test experiments
Default settings are moderate for a laptop/Colab run. For final paper plots, increase `MC_RUNS` to 30--50.

In [ ]:
MC_RUNS = 12
BASE_N = 60
experiments = {}
experiments["n_requests"] = run_mc("n_requests", [20,40,60,80,100], mc_runs=MC_RUNS, base_seed=10)
experiments["incompat_density"] = run_mc("incompat_density", [0.0,0.15,0.30,0.45,0.60], n_requests=BASE_N, mc_runs=MC_RUNS, base_seed=20)
experiments["llm_error"] = run_mc("llm_error", [0.00,0.08,0.16,0.24,0.32], n_requests=BASE_N, mc_runs=MC_RUNS, base_seed=30)
experiments["geofence_count"] = run_mc("geofence_count", [0,3,6,9,12], n_requests=BASE_N, mc_runs=MC_RUNS, base_seed=40)
experiments["deadline_tightness"] = run_mc("deadline_tightness", [0.30,0.45,0.60,0.75,0.90], n_requests=BASE_N, mc_runs=MC_RUNS, base_seed=50)
experiments["hazard_mix"] = run_mc("hazard_mix", [0.10,0.30,0.50,0.70,0.90], n_requests=BASE_N, mc_runs=MC_RUNS, base_seed=60)
all_results = pd.concat(experiments.values(), ignore_index=True)
all_results.head(), all_results.shape

## Twelve paper-relevant plots
These plots support the claim that HNP improves semantic safety and total mission cost under stress, not necessarily raw distance.

In [ ]:
plot_metric(experiments["n_requests"], "semantic_cost", "Total semantic mission cost vs. request count", "Semantic mission cost", "Number of requests")
plot_metric(experiments["incompat_density"], "hard_feasible", "Hard-feasibility rate vs. incompatibility density", "Feasibility rate", "Incompatibility density")
plot_metric(experiments["incompat_density"], "compat_viol", "Compatibility violations vs. incompatibility density", "Violations per route", "Incompatibility density")
plot_metric(experiments["llm_error"], "total_viol", "Total violations vs. LLM extraction error", "Violations per route", "LLM extraction error probability")
plot_metric(experiments["llm_error"], "verifier_recovery", "Verifier/fallback recovery rate vs. LLM extraction error", "Recovery rate", "LLM extraction error probability", algos=["HNP","HNP-NoVerify","HNP-NoCompat","HNP-NoRefine"])
plot_metric(experiments["geofence_count"], "geo_viol", "Geofence violations vs. restricted-zone density", "Geofence violations per route", "Number of geofenced zones")
plot_metric(experiments["deadline_tightness"], "lateness", "Deadline lateness vs. deadline tightness", "Weighted lateness", "Deadline tightness")
plot_metric(experiments["hazard_mix"], "hard_feasible", "Feasibility rate vs. hazardous/sensitive cargo mix", "Feasibility rate", "Hazardous/sensitive cargo probability")
plot_metric(experiments["n_requests"], "distance", "Raw travel distance vs. request count", "Distance", "Number of requests")
plot_metric(experiments["n_requests"], "runtime", "Runtime scalability vs. request count", "Runtime (s)", "Number of requests")
plot_metric(experiments["n_requests"], "energy", "Energy consumption vs. request count", "Energy proxy", "Number of requests")
plot_metric(experiments["geofence_count"], "noise", "Noise exposure vs. urban restriction density", "Noise exposure proxy", "Number of geofenced zones")

## Ablation summary table
The table below aggregates the stress case with incompatibility density 0.45.

In [ ]:
stress_df = experiments["incompat_density"]
stress_case = stress_df[np.isclose(stress_df["x"].astype(float), 0.45)]
summary_metrics = ["semantic_cost","distance","hard_feasible","compat_viol","geo_viol","lateness","runtime"]
summary_rows=[]
for algo,g in stress_case.groupby("algo"):
    row={"Algorithm":algo}
    for metric in summary_metrics:
        mean,ci=summarize_ci(g[metric].values); row[f"{metric}_mean"]=mean; row[f"{metric}_ci95"]=ci
    summary_rows.append(row)
summary_table=pd.DataFrame(summary_rows).sort_values("semantic_cost_mean")
summary_table

## Safety--efficiency trade-off relative to MPDD
A reviewer-safe plot showing that HNP can trade small distance overhead for large violation reduction.

In [ ]:
trade_rows=[]
for mc,gmc in stress_case.groupby("mc"):
    base=gmc[gmc["algo"]=="MPDD"]
    if base.empty: continue
    base=base.iloc[0]
    for _,row in gmc.iterrows():
        trade_rows.append({"algo":row["algo"],"extra_distance_vs_MPDD":row["distance"]-base["distance"],"violation_reduction_vs_MPDD":base["total_viol"]-row["total_viol"]})
trade_df=pd.DataFrame(trade_rows)
fig,ax=plt.subplots()
for algo in ALGORITHMS:
    sub=trade_df[trade_df["algo"]==algo]
    if not sub.empty: ax.scatter(sub["extra_distance_vs_MPDD"], sub["violation_reduction_vs_MPDD"], label=algo, alpha=0.7)
ax.axvline(0, linewidth=1); ax.axhline(0, linewidth=1)
ax.set_title("Safety-efficiency trade-off relative to MPDD")
ax.set_xlabel("Extra distance relative to MPDD")
ax.set_ylabel("Violation reduction relative to MPDD")
ax.legend(); plt.tight_layout()

In [ ]:
all_results.to_csv("semantic_uav_stress_results.csv", index=False)
summary_table.to_csv("semantic_uav_ablation_summary.csv", index=False)
print("Saved semantic_uav_stress_results.csv and semantic_uav_ablation_summary.csv")

## Interpretation guidance
- HNP may not have the shortest raw distance; MPDD and NN-PDP ignore semantic constraints.
- The main claims should be built on semantic mission cost, feasibility, compatibility safety, geofence compliance, robustness to LLM errors, and ablation trends.
- HNP-NoVerify isolates the benefit of verification.
- HNP-NoCompat isolates the benefit of compatibility-aware routing.
- HNP-NoRefine isolates the benefit of source-anchored refinement.